In [ ]:
# ============================================================
# 0) Install (Colab)
# ============================================================
!pip -q install -U "numpy>=2"
!pip -q install datasets huggingface_hub jiwer torch transformers accelerate omnilingual-asr tqdm pandas soundfile librosa
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# (Optional but common for OmniASR setups) Ensure compatible torch stack.
# Comment these out if your runtime already has a working torch+cuda combo.
!pip -q uninstall -y torch torchvision torchaudio fairseq2
!pip -q install --no-cache-dir \
  torch==2.8.0+cu126 torchvision==0.23.0+cu126 torchaudio==2.8.0+cu126 \
  --index-url https://download.pytorch.org/whl/cu126
!pip -q install --no-cache-dir fairseq2 \
  --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu126


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 135.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.5/427.5 kB 15.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 1. Uninstall existing conflicting packages to ensure a clean slate
!pip uninstall -y torch torchvision torchaudio fairseq2 torchcodec omnilingual-asr

# 2. Install system-level FFmpeg
# Colab's default FFmpeg (v4) works fine with the stable PyTorch stack.
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# 3. Install Stable PyTorch 2.5.1 (CUDA 12.4)
# We switch from 2.8.0 to 2.5.1 to avoid the "undefined symbol" ABI errors.
!pip install --no-cache-dir \
  torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 \
  --index-url https://download.pytorch.org/whl/cu124

# 4. Install Fairseq2 compatible with PyTorch 2.5.1
# Note the URL changes from pt2.8.0/cu126 to pt2.5.1/cu124
!pip install --no-cache-dir fairseq2 \
  --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.5.1/cu124

# 5. Install remaining libraries
!pip -q install -U "numpy>=2"
!pip -q install datasets huggingface_hub jiwer tqdm pandas soundfile librosa omnilingual-asr

# 6. Environment Fix for Colab
# Sometimes Colab hides the CUDA libraries from ffmpeg/torchcodec.
# This ensures they are found.
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib/python3.12/site-packages/torchlib:' + os.environ.get('LD_LIBRARY_PATH', '')

print("Installation complete. Proceed to Section 8.")

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
Found existing installation: fairseq2 0.8.1
Uninstalling fairseq2-0.8.1:
  Successfully uninstalled fairseq2-0.8.1
Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
Found existing installation: omnilingual-asr 0.1.0
Uninstalling omnilingual-asr-0.1.0:
  Successfully uninstalled omnilingual-asr-0.1.0
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Looking in

In [ ]:
# ============================================================
# FULL INSTALLATION SCRIPT - Colab (fixes libavutil.so missing)
# ============================================================

# ------------------------------------------------------------
# Step 1: Remove any broken ffmpeg and install from PPA
#         that ships actual shared .so libraries
# ------------------------------------------------------------
!apt-get -qq remove -y ffmpeg
!apt-get -qq autoremove -y
!add-apt-repository -y ppa:ubuntustudio-ppa/backports 2>/dev/null || true
!apt-get -qq update

# Install FFmpeg 6.x with full shared library support
!apt-get -qq install -y \
  ffmpeg \
  libavcodec60 \
  libavformat60 \
  libavutil58 \
  libswscale7 \
  libavdevice60 \
  libavfilter9 \
  libavcodec-dev \
  libavformat-dev \
  libavutil-dev \
  libswscale-dev

# ------------------------------------------------------------
# Step 2: Find where libavutil actually lives and symlink
#         to every version TorchCodec tries to load (.57-.60)
# ------------------------------------------------------------
import subprocess, glob, os

result = subprocess.run(
    ["find", "/usr", "-name", "libavutil.so*", "-not", "-type", "d"],
    capture_output=True, text=True
)
print("Found libavutil files:\n", result.stdout)

# Create symlinks for all versions TorchCodec probes
# !find /usr -name "libavutil.so*" | head -5
# !ldconfig

# Manually symlink if needed — covers versions 57, 58, 59, 60
!bash -c '
BASE=$(find /usr/lib /usr/local/lib -name "libavutil.so.*" -not -type l 2>/dev/null | head -1)
if [ -z "$BASE" ]; then
  echo "ERROR: libavutil.so not found anywhere"
  exit 1
fi
echo "Base library: $BASE"
DIR=$(dirname $BASE)
for VER in 56 57 58 59 60; do
  TARGET="$DIR/libavutil.so.$VER"
  if [ ! -f "$TARGET" ] && [ ! -L "$TARGET" ]; then
    ln -sf "$BASE" "$TARGET"
    echo "Created symlink: $TARGET -> $BASE"
  else
    echo "Already exists: $TARGET"
  fi
done
ldconfig
'

# Also symlink the other required libs
!bash -c '
for LIB in libavcodec libavformat libswscale libavdevice libavfilter; do
  BASE=$(find /usr/lib /usr/local/lib -name "${LIB}.so.*" -not -type l 2>/dev/null | head -1)
  if [ -n "$BASE" ]; then
    DIR=$(dirname $BASE)
    echo "Found $LIB base: $BASE"
    for VER in 56 57 58 59 60 61; do
      TARGET="$DIR/${LIB}.so.$VER"
      if [ ! -f "$TARGET" ] && [ ! -L "$TARGET" ]; then
        ln -sf "$BASE" "$TARGET"
      fi
    done
  fi
done
ldconfig
'

# ------------------------------------------------------------
# Step 3: Wipe conflicting torch/codec packages
# ------------------------------------------------------------
!pip -q uninstall -y \
  torch torchvision torchaudio \
  torchcodec fairseq2 \
  omnilingual-asr 2>/dev/null || true

# ------------------------------------------------------------
# Step 4: Install PyTorch 2.5.1 + CUDA 12.1 (stable, tested)
# ------------------------------------------------------------
!pip -q install --no-cache-dir \
  torch==2.5.1+cu121 \
  torchvision==0.20.1+cu121 \
  torchaudio==2.5.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# ------------------------------------------------------------
# Step 5: Install fairseq2 matching PT 2.5.1 + CU121
# ------------------------------------------------------------
!pip -q install --no-cache-dir fairseq2 \
  --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.5.1/cu121

# ------------------------------------------------------------
# Step 6: Install TorchCodec 0.2.1 (matches PT 2.5.x)
# ------------------------------------------------------------
!pip -q install --no-cache-dir \
  "torchcodec==0.2.1" \
  --extra-index-url https://download.pytorch.org/whl/cu121

# ------------------------------------------------------------
# Step 7: Core libraries
# ------------------------------------------------------------
!pip -q install -U "numpy>=2"
!pip -q install --no-cache-dir \
  datasets \
  huggingface_hub \
  jiwer \
  transformers \
  accelerate \
  tqdm \
  pandas \
  soundfile \
  librosa

# ------------------------------------------------------------
# Step 8: OmniASR without deps (prevents torch version clobber)
# ------------------------------------------------------------
!pip -q install omnilingual-asr --no-deps

# ------------------------------------------------------------
# Step 9: Reload linker cache and verify everything
# ------------------------------------------------------------
!ldconfig
!echo "=== libavutil symlinks ==="
!find /usr/lib /usr/local/lib -name "libavutil.so*" 2>/dev/null | sort

import sys
print("\n=== Python ===", sys.version)

try:
    import torch
    print("=== PyTorch ===", torch.__version__)
    print("   CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("   CUDA device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("❌ PyTorch:", e)

try:
    import torchcodec
    print("=== TorchCodec ===", torchcodec.__version__, "✓")
except Exception as e:
    print("❌ TorchCodec:", e)

try:
    import fairseq2
    print("=== fairseq2 ===", fairseq2.__version__, "✓")
except Exception as e:
    print("❌ fairseq2:", e)

try:
    import transformers
    print("=== Transformers ===", transformers.__version__, "✓")
except Exception as e:
    print("❌ Transformers:", e)

try:
    result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    print("=== FFmpeg ===", result.stdout.split("\n")[0], "✓")
except Exception as e:
    print("❌ FFmpeg:", e)

print("\n✅ Installation complete — check for any ❌ above.")

SyntaxError: unterminated string literal (detected at line 52) (4080083889.py, line 52)

In [ ]:
import os
import re
import time
import unicodedata
from dataclasses import dataclass
from typing import Callable, Dict, Optional, List, Any
from huggingface_hub import hf_hub_download

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset, Audio

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM(GB):", round(props.total_memory/1e9, 1))


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB | VRAM(GB): 42.4


In [ ]:
from huggingface_hub import login

# Attempt to log in using the HF_TOKEN secret
try:
    login()
    print("Successfully logged in to Hugging Face.")
except Exception as e:
    print(f"Could not log in to Hugging Face. Ensure HF_TOKEN is set as a Colab secret. Error: {e}")

Successfully logged in to Hugging Face.


In [ ]:


# ============================================================
# 1) Config
# ============================================================
DATASET_ID = "AhmedAshrafMarzouk/bulbul-msa-dz"  # <-- change as needed
AUDIO_COL  = "Audio"                                       # dataset audio column name
TEXT_COL   = "text"                                        # dataset text column name

OUT_DIR = "./asr_runs_omniasr_v3"
os.makedirs(OUT_DIR, exist_ok=True)

# OmniASR models (HF repo id -> family)
MODELS = {
    "facebook/omniASR-LLM-300M": "omni_llm",
    # "facebook/omniASR-LLM-1B":   "omni_llm",
    # "facebook/omniASR-LLM-3B":   "omni_llm",
    # "facebook/omniASR-LLM-7B":   "omni_llm",
}

# Choose models to run in this notebook
MODEL_SET = [
    "facebook/omniASR-LLM-300M",
    "facebook/omniASR-LLM-1B",
    "facebook/omniASR-LLM-3B",
    "facebook/omniASR-LLM-7B",
]

# Experiment knobs
MAX_SAMPLES_PER_SPLIT: Optional[int] = None  # None = full split; e.g., 100 for quick run
SHOW_EXAMPLES = 2
TARGET_SR = 16000

REPORT_PATH = os.path.join(OUT_DIR, "report.csv")



In [ ]:

# ============================================================
# 2) Arabic normalization
# ============================================================
ARABIC_BLOCKS = r"\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF"

def normalize_ar(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u0640", "")  # tatweel

    # remove combining marks (diacritics)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    # Arabic-Indic numbers -> ASCII
    for i, d in enumerate("٠١٢٣٤٥٦٧٨٩"):
        text = text.replace(d, str(i))

    # Alef variants
    text = re.sub(r"[أإآ]", "ا", text)

    # remove punctuation/symbols (keep Arabic + ascii word chars + whitespace)
    text = re.sub(rf"[^{ARABIC_BLOCKS}\w\s]", " ", text)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text



In [ ]:

# ============================================================
# 3) Robust audio loading + resampling to 16k
# ============================================================
def get_audio_file(ex: dict) -> str:
    if "audio_file" in ex and ex["audio_file"] is not None:
        return str(ex["audio_file"])
    a = ex.get(AUDIO_COL, {})
    if isinstance(a, dict) and a.get("path"):
        return str(a["path"])
    return ""

def get_waveform_and_sr(ex: dict, audio_key: str, default_sr: int) -> tuple[np.ndarray, int]:
    """
    Robust loader for HF datasets where audio is stored in the dataset repo.
    Works with:
      - decoded Audio dict: {"array","sampling_rate"}
      - decode=False dict: {"path"} (often just a filename) and/or {"bytes"}
      - string filename columns (rare but happens)
    If path is not local, it downloads via hf_hub_download(repo_type="dataset").
    """
    a = ex[audio_key]

    # Case 0: column is a string filename (some datasets do this)
    if isinstance(a, str):
        rel_path = a
        local_path = hf_hub_download(
            repo_id=DATASET_ID,
            filename=rel_path,
            repo_type="dataset",
        )
        import soundfile as sf
        wf, sr = sf.read(local_path, dtype="float32")
        if wf.ndim > 1:
            wf = wf.mean(axis=1)
        return np.asarray(wf, dtype=np.float32), int(sr)

    # already array-like
    if isinstance(a, (list, tuple, np.ndarray)):
        wf = np.asarray(a, dtype=np.float32)
        if wf.ndim > 1:
            wf = wf.mean(axis=-1)
        return wf, default_sr

    if not isinstance(a, dict):
        raise TypeError(f"Unsupported audio format in `{audio_key}`: type={type(a)}")

    # Case 1: decoded format
    if "array" in a and a["array"] is not None:
        wf = np.asarray(a["array"], dtype=np.float32)
        sr = int(a.get("sampling_rate", default_sr))
        if wf.ndim > 1:
            wf = wf.mean(axis=-1)
        return wf, sr

    # Case 2: bytes present (best)
    if "bytes" in a and a["bytes"]:
        import io
        import soundfile as sf
        wf, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
        if wf.ndim > 1:
            wf = wf.mean(axis=1)
        return np.asarray(wf, dtype=np.float32), int(sr)

    # Case 3: path-like (may be local path OR just a filename in HF repo)
    if "path" in a and a["path"]:
        path = a["path"]

        # 3a) If it's already a real local file, just read it
        if os.path.exists(path):
            import soundfile as sf
            wf, sr = sf.read(path, dtype="float32")
            if wf.ndim > 1:
                wf = wf.mean(axis=1)
            return np.asarray(wf, dtype=np.float32), int(sr)

        # 3b) Otherwise treat it as a relative filename in the dataset repo and download
        rel_path = path  # e.g. "salama_00f0fdd_ae-ad_f_148.wav" or "audio/xxx.wav"
        local_path = hf_hub_download(
            repo_id=DATASET_ID,
            filename=rel_path,
            repo_type="dataset",
        )

        import soundfile as sf
        wf, sr = sf.read(local_path, dtype="float32")
        if wf.ndim > 1:
            wf = wf.mean(axis=1)
        return np.asarray(wf, dtype=np.float32), int(sr)

    raise KeyError(f"Could not decode audio. Keys present: {list(a.keys())}")

def ensure_16k(wf: np.ndarray, sr: int, target_sr: int = TARGET_SR) -> tuple[np.ndarray, int]:
    if sr == target_sr:
        return wf.astype(np.float32), sr
    import librosa
    wf16 = librosa.resample(wf.astype(np.float32), orig_sr=sr, target_sr=target_sr)
    return wf16.astype(np.float32), target_sr



In [ ]:

# ============================================================
# 4) Metrics (WER, CER, NWER, NCER) + report writer
# ============================================================
def _tokenize_words(s: str) -> List[str]:
    s = (s or "").strip()
    return s.split() if s else []

def _tokenize_chars(s: str) -> List[str]:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace(" ", "")  # ignore spaces for CER
    return list(s) if s else []

def _edit_distance(ref_tokens: List[str], hyp_tokens: List[str]) -> int:
    n, m = len(ref_tokens), len(hyp_tokens)
    if n == 0:
        return m
    if m == 0:
        return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = cur
    return dp[m]

def _error_rate(refs: List[str], hyps: List[str], tokenizer: Callable[[str], List[str]]) -> float:
    total_edits = 0
    total_ref_tokens = 0
    for r, h in zip(refs, hyps):
        rt = tokenizer(r)
        ht = tokenizer(h)
        total_edits += _edit_distance(rt, ht)
        total_ref_tokens += len(rt)

    if total_ref_tokens == 0:
        return 0.0 if all((not (h or "").strip()) for h in hyps) else 1.0
    return total_edits / total_ref_tokens

def compute_metrics(refs: List[str], hyps: List[str]) -> Dict[str, float]:
    WER = _error_rate(refs, hyps, _tokenize_words)
    CER = _error_rate(refs, hyps, _tokenize_chars)

    refs_n = [normalize_ar(r) for r in refs]
    hyps_n = [normalize_ar(h) for h in hyps]
    NWER = _error_rate(refs_n, hyps_n, _tokenize_words)
    NCER = _error_rate(refs_n, hyps_n, _tokenize_chars)

    return {"WER": float(WER), "CER": float(CER), "NWER": float(NWER), "NCER": float(NCER)}

def append_report_row(report_path: str, row: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    if os.path.exists(report_path):
        df = pd.read_csv(report_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(report_path, index=False, encoding="utf-8")


In [ ]:

# ============================================================
# 5) OmniASR transcriber
# ============================================================
def make_omniasr_transcriber(model_id: str, lang_code: str = "arb_Arab", batch_size: int = 1) -> Callable:
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    hf_to_card = {
        "facebook/omniASR-LLM-300M": "omniASR_LLM_300M",
        "facebook/omniASR-LLM-1B":   "omniASR_LLM_1B",
        "facebook/omniASR-LLM-3B":   "omniASR_LLM_3B",
        "facebook/omniASR-LLM-7B":   "omniASR_LLM_7B",
    }
    model_card = hf_to_card.get(model_id)
    if model_card is None:
        raise ValueError(f"Unknown OmniASR model_id: {model_id}")

    pipeline = ASRInferencePipeline(model_card=model_card, device=device, dtype=dtype)

    def _to_item(wf: np.ndarray, sample_rate: int) -> Dict[str, Any]:
        # OmniASR wants a dict with waveform (tensor) and sample_rate
        t = torch.tensor(wf, dtype=torch.float32, device="cpu")
        if t.ndim == 2:
            t = t.mean(dim=0)
        elif t.ndim != 1:
            t = t.reshape(-1)
        return {"waveform": t, "sample_rate": int(sample_rate)}

    def transcribe(wf: np.ndarray, sample_rate: int = TARGET_SR, lang: Optional[str] = None) -> str:
        lc = lang if lang is not None else lang_code
        item = _to_item(wf, sample_rate)
        outs = pipeline.transcribe([item], lang=[lc], batch_size=batch_size)
        return (outs[0] if outs else "").strip()

    return transcribe


def make_transcriber(model_id: str) -> Callable:
    fam = MODELS[model_id]
    if fam == "omni_llm":
        return make_omniasr_transcriber(model_id, lang_code="arb_Arab", batch_size=1)
    raise ValueError(f"Unknown family: {fam}")



In [ ]:

# ============================================================
# 6) Dataset split helpers
# ============================================================
def pick_splits(ds_dict) -> Dict[str, str]:
    """
    Returns mapping like {"dev": "<actual_split>", "test": "<actual_split>"}.
    Works with dev/validation/val and test.
    """
    def first_existing(cands: List[str]) -> Optional[str]:
        for c in cands:
            if c in ds_dict:
                return c
        return None

    dev_name = first_existing(["dev", "validation", "valid", "val", "train"])
    test_name = first_existing(["test", "eval", "evaluation"])

    out = {}
    if dev_name is not None:
        out["dev"] = dev_name
    if test_name is not None:
        out["test"] = test_name
    if not out:
        raise ValueError(f"Could not find dev/test-like splits in: {list(ds_dict.keys())}")
    return out


In [ ]:


# ============================================================
# 7) Core experiment loop (pred CSV + report.csv with WER/CER/NWER/NCER)
# ============================================================
@dataclass
class SplitResult:
    model_id: str
    split: str
    n: int
    WER: float
    CER: float
    NWER: float
    NCER: float
    seconds: float
    sec_per_utt: float
    preds_csv: str

def run_split(
    ds_split,
    model_id: str,
    split_name: str,
    transcribe_fn: Callable,
    max_samples: Optional[int] = None,
) -> SplitResult:
    if max_samples is not None:
        ds_split = ds_split.select(range(min(max_samples, len(ds_split))))

    refs: List[str] = []
    hyps: List[str] = []
    rows: List[Dict[str, Any]] = []

    t0 = time.time()
    pbar = tqdm(range(len(ds_split)), desc=f"{split_name} | {model_id}", leave=False)

    for i in pbar:
        ex = ds_split[i]

        wf, sr = get_waveform_and_sr(ex, audio_key=AUDIO_COL, default_sr=TARGET_SR)
        wf, sr = ensure_16k(wf, sr, target_sr=TARGET_SR)

        ref = ex.get(TEXT_COL, "")
        hyp = transcribe_fn(wf, sample_rate=sr)

        refs.append(str(ref) if ref is not None else "")
        hyps.append(str(hyp) if hyp is not None else "")

        rows.append({
            "audio_file": get_audio_file(ex),
            "text": refs[-1],
            "predicted": hyps[-1],
        })

        if i < SHOW_EXAMPLES:
            pbar.write(f"[{split_name} ex {i}]")
            pbar.write(f"  REF : {refs[-1]}")
            pbar.write(f"  HYP : {hyps[-1]}")
            pbar.write(f"  NREF: {normalize_ar(refs[-1])}")
            pbar.write(f"  NHYP: {normalize_ar(hyps[-1])}")

    dt = time.time() - t0
    metrics = compute_metrics(refs, hyps)

    preds_csv = os.path.join(OUT_DIR, f"preds_{split_name}_{model_id.replace('/', '_')}.csv")
    pd.DataFrame(rows).to_csv(preds_csv, index=False, encoding="utf-8")

    result = SplitResult(
        model_id=model_id,
        split=split_name,
        n=len(rows),
        WER=metrics["WER"],
        CER=metrics["CER"],
        NWER=metrics["NWER"],
        NCER=metrics["NCER"],
        seconds=float(dt),
        sec_per_utt=float(dt / max(1, len(rows))),
        preds_csv=preds_csv,
    )

    # Append to master report
    append_report_row(REPORT_PATH, {
        "model_id": result.model_id,
        "split": result.split,
        "n": result.n,
        "WER": result.WER,
        "CER": result.CER,
        "NWER": result.NWER,
        "NCER": result.NCER,
        "seconds": result.seconds,
        "sec_per_utt": result.sec_per_utt,
        "preds_csv": result.preds_csv,
        "dataset_id": DATASET_ID,
    })

    return result


def run_models(dataset_id: str, model_ids: List[str]) -> pd.DataFrame:
    ds = load_dataset(dataset_id)
    ds = ds.cast_column(AUDIO_COL, Audio(decode=False))  # <-- avoids torchcodec

    split_map = pick_splits(ds)
    print("Available splits:", list(ds.keys()))
    print("Using split map :", split_map)

    all_results: List[Dict[str, Any]] = []

    for model_id in model_ids:
        print("\n" + "=" * 90)
        print("Model:", model_id)

        transcribe_fn = make_transcriber(model_id)

        for logical_split, actual_split in split_map.items():
            r = run_split(
                ds_split=ds[actual_split],
                model_id=model_id,
                split_name=logical_split,
                transcribe_fn=transcribe_fn,
                max_samples=MAX_SAMPLES_PER_SPLIT,
            )
            all_results.append(r.__dict__)

    df = pd.DataFrame(all_results).sort_values(["model_id", "split"])
    return df


In [ ]:


# ============================================================
# 8) Run
# ============================================================
results_df = run_models(DATASET_ID, MODEL_SET)
print("\n=== Summary (this run) ===")
display(results_df)

print("\nMaster report saved at:", REPORT_PATH)
display(pd.read_csv(REPORT_PATH).tail(20))



parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   535/535 100% 0:00:00

test | facebook/omniASR-LLM-300M:   0%|          | 0/55 [00:00<?, ?it/s]

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.8.0+cu126) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call

In [ ]:

# ============================================================
# 9) Post-hoc: report from a predictions CSV (WER/CER/NWER/NCER)
# ============================================================
def report_from_predictions_csv(csv_path: str, text_col: str = "text", pred_col: str = "predicted") -> Dict[str, float]:
    df = pd.read_csv(csv_path)
    refs = df[text_col].fillna("").astype(str).tolist()
    hyps = df[pred_col].fillna("").astype(str).tolist()
    m = compute_metrics(refs, hyps)

    print("=== Report from CSV ===")
    print("csv :", csv_path)
    print("n   :", len(df))
    print("WER :", m["WER"])
    print("CER :", m["CER"])
    print("NWER:", m["NWER"])
    print("NCER:", m["NCER"])
    return m

# Example:
report_from_predictions_csv("/content/preds_dev_facebook_omniASR-LLM-7B.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')